In [22]:
import io
import torch
import open_clip
from PIL import Image

# Initialize the tokenizer for OpenCLIP ViT-H/14
tokenizer = open_clip.get_tokenizer("ViT-H-14")

In [1]:
import pandas as pd

df = pd.read_parquet(rf"E:\Data\query-earth\git-10mil\git_10m_1000000_random.parquet")

In [12]:
import open_clip
import torch
from torchvision import transforms

ckpt_path = rf"E:\Weights\vlm\RS5M_ViT-H-14.pt"

model, _, _ = open_clip.create_model_and_transforms("ViT-H/14")
checkpoint = torch.load(ckpt_path, map_location="cpu")
msg = model.load_state_dict(checkpoint, strict=False)
model = model.to("cuda")

In [13]:
def _convert_to_rgb(image):
    return image.convert('RGB')

normalize = transforms.Normalize(
            mean=[0.48145466, 0.4578275, 0.40821073], std=[0.26862954, 0.26130258, 0.27577711]
        )
image_preprocess = transforms.Compose([
            transforms.Resize(
                size=224,
                interpolation=transforms.InterpolationMode.BICUBIC,
            ),
            transforms.CenterCrop(224),
            _convert_to_rgb,
            transforms.ToTensor(),
            normalize,
        ])

In [23]:
from PIL import Image
import io

def get_image_text_scores(df, row_idx, prompts, model, img_preprocess, img_col = "image_bytes"):
    """Calculates similarity scores between an image byte string in a DataFrame row and a list of text prompts.

    Args:
        df (pd.DataFrame): Your DataFrame containing image bytes.
        row_idx (int): Row index or positional index.
        img_col (str): Column name containing the image bytes.
        prompts (list of str): List of text prompts to score against the image.
        model: Loaded OpenCLIP model on CUDA.
        img_preprocess: Image preprocessing function.

    Returns:
        dict: Mapping of {prompt: score}
    """
    # 1. Load image from bytes
    img_bytes = df.iloc[row_idx][img_col]
    image = Image.open(io.BytesIO(img_bytes)).convert("RGB")

    # 2. Preprocess image & move to device
    image_tensor = img_preprocess(image).unsqueeze(0).to("cuda")

    # 3. Tokenize prompts & move to device
    text_tokens = tokenizer(prompts).to("cuda")

    # 4. Compute embeddings without tracking gradients
    with torch.no_grad(), torch.cuda.amp.autocast():
        image_features = model.encode_image(image_tensor)
        text_features = model.encode_text(text_tokens)

        # Normalize features to unit vectors
        image_features /= image_features.norm(dim=-1, keepdim=True)
        text_features /= text_features.norm(dim=-1, keepdim=True)

        # Calculate cosine similarity (scaled by model logit_scale if preferred)
        # Using raw cosine similarity mapped to 0-100 range:
        similarity_scores = (image_features @ text_features.T).squeeze(0)

    # 5. Pair prompts with scores
    scores = {
        prompt: score.item() for prompt, score in zip(prompts, similarity_scores)
    }
    return scores

In [26]:
from PIL import Image 
import io

row = 10
image_bytes = df.at[row, 'image_bytes']
image = Image.open(io.BytesIO(image_bytes))
image.show()

In [24]:
get_image_text_scores(df, 10, ['tree', 'solar panels', 'no solar panels', "red houses"], model, image_preprocess)

C:\Users\sus14836\AppData\Local\Temp\ipykernel_12384\790850455.py:29: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast():


{'tree': 0.1695556640625,
 'solar panels': 0.16162109375,
 'no solar panels': 0.1646728515625,
 'red houses': 0.1986083984375}